In [3]:
#transformers library and torch
!pip install transformers torch

In [4]:
from transformers import AutoTokenizer

model_name = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

text = "I love learning machine learning!"

# 4. Convert text to tokens
tokens = tokenizer.tokenize(text)
print(f"Tokens: {tokens}")

ids = tokenizer.convert_tokens_to_ids(tokens)
print(f"Token IDs: {ids}")

Tokens: ['i', 'love', 'learning', 'machine', 'learning', '!']
Token IDs: [1045, 2293, 4083, 3698, 4083, 999]


In [ ]:
from transformers import pipeline

# We use a pipeline for 'sentiment-analysis' (which is basically toxic vs non-toxic)
classifier = pipeline("sentiment-analysis")
#classifier = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")


# Test it with different sentences
results = classifier([
    "I love this project, it is amazing!",
    "Oh great, another error in my code.",
    "I hate this, it is the worst thing ever.",
    "Oh great, another error in my code.",
    "You slayed that presentation! 👏"
])

for result in results:
    print(f"Label: {result['label']}, Score: {result['score']:.4f}")

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

Label: POSITIVE, Score: 0.9999
Label: NEGATIVE, Score: 0.9980
Label: NEGATIVE, Score: 0.9997
Label: POSITIVE, Score: 0.9999
Label: NEGATIVE, Score: 0.9998
Label: NEGATIVE, Score: 0.9980
Label: NEGATIVE, Score: 0.9683


In [7]:
!pip install transformers[torch] datasets evaluate

In [6]:
import pandas as pd
from datasets import Dataset
from transformers import AutoTokenizer

df = pd.read_csv('train.csv')

df = df.sample(30000, random_state=42)

data = df[['comment_text', 'toxic']]
data.columns = ['text', 'label']

#
positive_slang = [
    ("You slayed that presentation! 👏", 0),
    ("She absolutely slays in that outfit!", 0),
    ("Slay queen!", 0),
    ("That performance was a total slay", 0),
    ("You are slaying the game right now", 0),
    ("I'm slaying my exams this week", 0),
    ("This makeup look slays", 0),
    ("Your art style slays", 0),
    ("Absolutely slayed the choreography", 0),
    ("That vocal run slayed me", 0),
]
slang_df = pd.DataFrame(positive_slang, columns=['text', 'label'])

data = pd.concat([data, slang_df], ignore_index=True)

#Convert the FIXED data to Hugging Face Dataset format
dataset = Dataset.from_pandas(data)

#Split into 80% training, 20% validation
dataset = dataset.train_test_split(test_size=0.2)
train_dataset = dataset['train']
test_dataset = dataset['test']

#Tokenize the data
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)

tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)

print("Data is loaded, fixed with slang, and tokenized! Ready for training.")

Map:   0%|          | 0/24008 [00:00<?, ? examples/s]

Map:   0%|          | 0/6002 [00:00<?, ? examples/s]

Data is loaded, fixed with slang, and tokenized! Ready for training.


In [8]:
!pip install evaluate

In [9]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
import numpy as np
import evaluate

# num_labels=2 because it's either Toxic (1) or Not Toxic (0)
model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)

metric = evaluate.load("accuracy")
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch", # Check accuracy after every epoch
    per_device_train_batch_size=16,
    num_train_epochs=3,          # Go through the data 3 times
    weight_decay=0.01,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    compute_metrics=compute_metrics,
)

trainer.train()

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy
1,0.111250,0.092618,0.963346
2,0.057635,0.133643,0.962846
3,0.017712,0.181019,0.962679


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=4503, training_loss=0.07242159612650609, metrics={'train_runtime': 3843.6634, 'train_samples_per_second': 18.738, 'train_steps_per_second': 1.172, 'total_flos': 9540831920799744.0, 'train_loss': 0.07242159612650609, 'epoch': 3.0})

In [15]:
from transformers import pipeline

#pipeline using trained model
my_toxic_classifier = pipeline("text-classification", model=model, tokenizer=tokenizer)

test_comments = [
    "I love you so much!",
    "You are a complete idiot and I hate you.",
    "This is a decent review of the movie.",
    "I love this project, it is amazing!",
    "Oh great, another error in my code.",
    "I hate this, it is the worst thing ever.",
    "Oh great, another error in my code.",
    "You slayed that presentation! 👏",
    "I'm gonna beat you.",
    "I'm going to beat you up!"
]

results = my_toxic_classifier(test_comments)
for text, res in zip(test_comments, results):
    print(f"Text: {text} | Result: {res['label']} | Score: {res['score']:.4f}")

Text: I love you so much! | Result: LABEL_0 | Score: 0.9994
Text: You are a complete idiot and I hate you. | Result: LABEL_1 | Score: 0.9987
Text: This is a decent review of the movie. | Result: LABEL_0 | Score: 0.9999
Text: I love this project, it is amazing! | Result: LABEL_0 | Score: 0.9999
Text: Oh great, another error in my code. | Result: LABEL_0 | Score: 0.9999
Text: Your paper is bullshit. | Result: LABEL_1 | Score: 0.9986
Text: Nigga, you are really nice. | Result: LABEL_1 | Score: 0.9982
Text: I hate this, it is the worst thing ever. | Result: LABEL_0 | Score: 0.6979
Text: Oh great, another error in my code. | Result: LABEL_0 | Score: 0.9999
Text: You slayed that presentation! 👏 | Result: LABEL_0 | Score: 0.9999
Text: I love you and hate you too. | Result: LABEL_0 | Score: 0.8701
Text: I'm gonna beat you. | Result: LABEL_0 | Score: 0.9994
Text: I'm going to beat you up! | Result: LABEL_0 | Score: 0.9995


In [17]:
# 1. LAYER 1: The Blacklist (Regex/Keywords)
#phrases that MUST be blocked, no matter what the AI thinks
BANNED_WORDS = ["racialslur1", "beat you up", "kill you", "punch you"]
def layer_1_blacklist(text):
    text_lower = text.lower()
    for word in BANNED_WORDS:
        if word in text_lower:
            return "BANNED" # Instant block, no AI needed
    return "PASS"

# 2. LAYER 2 & 3: The AI and Human Review
def moderate_comment(text):
    #1.Check Blacklist
    if layer_1_blacklist(text) == "BANNED":
        return "❌ REJECTED (Blacklist)"

    #2.Check AI Model
    result = my_toxic_classifier(text)[0]
    label = result['label']
    score = result['score']

    is_toxic = (label == "LABEL_1")

    #3.Human Review (Confidence Threshold)
    #If the model is only 50% to 80% sure, send to human
    #If the score is very high (e.g., > 0.8),trust the AI
    if 0.5 <= score <= 0.8:
        return "⚠️ SENT TO HUMAN REVIEW (Unsure)"

    if is_toxic:
        return f"❌ REJECTED (AI Confident: {score:.4f})"
    else:
        return f"✅ APPROVED (AI Confident: {score:.4f})"

# ==========================================
# TEST THE HYBRID SYSTEM
# ==========================================
test_cases = [
    "I love this project!",                # Should be Approved
    "You are a complete idiot!",           # Should be Rejected by AI
    "This is a weird comment maybe?",      # Might go to Human Review
    "Some sentence with racialslur1 here", # Should be Rejected by Blacklist
    "You slayed that presentation!",       # Test your fix!
    "I'm going to beat you up!"            # Test your fix!
]

for text in test_cases:
    print(f"Text: {text} | Result: {moderate_comment(text)}")

Text: I love this project! | Result: ✅ APPROVED (AI Confident: 0.9999)
Text: You are a complete idiot! | Result: ❌ REJECTED (AI Confident: 0.9987)
Text: This is a weird comment maybe? | Result: ✅ APPROVED (AI Confident: 0.9999)
Text: Some sentence with racialslur1 here | Result: ❌ REJECTED (Blacklist)
Text: You slayed that presentation! | Result: ✅ APPROVED (AI Confident: 0.9999)
Text: I'm going to beat you up! | Result: ❌ REJECTED (Blacklist)


In [ ]:
!pip install onnxruntime onnx
!pip install onnxscript

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# 2. We use the tokenizer. Since `model` is already in memory from trainer.train(),
# we DO NOT reload an untrained model base on DistilBERT.
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Ensure our trained model is in evaluation mode
model.eval()

text = "This is a test sentence"
inputs = tokenizer(text, return_tensors="pt", padding='max_length', truncation=True, max_length=512)
dummy_input = (inputs['input_ids'], inputs['attention_mask'])

torch.onnx.export(
    model,
    dummy_input,
    "toxic_model.onnx",
    export_params=True,
    opset_version=14, # Updated to 14 to avoid onnx warnings
    do_constant_folding=True,
    input_names=['input_ids', 'attention_mask'],
    output_names=['output'],
    dynamic_axes={'input_ids': {0: 'batch_size'}, 'attention_mask': {0: 'batch_size'}, 'output': {0: 'batch_size'}}
)

print("🚀 Success! Your FINETUNED model has been converted to toxic_model.onnx")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 689.1/689.1 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.1/164.1 kB 12.1 MB/s eta 0:00:00


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/tmp/ipykernel_6482/954369967.py:22: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is uns

[torch.onnx] Obtain model graph for `DistilBertForSequenceClassification([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `DistilBertForSequenceClassification([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...


/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/__init__.py", line 120, in call
    converted_proto = _c_api_utils.call_onnx_api(
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
             ^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/__init__.py", line 115, in _partial_convert_version
    return onnx.version_converter.convert_version(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnx/version_converter.py", line 39, in convert_version
    converted_model_str = C.convert_version(model_

[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
Applied 31 of general pattern rewrite rules.
🚀 Success! Your model has been converted to toxic_model.onnx
